In [ ]:
import NICE as nice
import pandas as pd

XXXX-1 = pd.read_csv('Initial_Data/winequality_red_6V8.csv')
label = 'Quality'

In [ ]:
# 4 parameter boundaries
bounds = [
    (2, 2),        # n: Number of features (integer)
    (0.85, 0.98),  # x: PSM threshold (continuous value)
    (5, 15),       # k_ir: Calculate the number of neighbors of IR_nn (integer)
    (10, 40)       # k_vote: Number of neighbors used for voting (integer)
]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
import numpy as np



#A parameter evaluation function is defined to optimize the hyperparameter configuration of the PSM-CFA algorithm, 
# with the balanced accuracy (G-mean) of the random forest classifier used as the evaluation metric.
def evaluate_params(params):
    n = int(np.round(params[0]))
    x = np.round(np.clip(params[1], 0.75, 0.95), 2)
    k_ir = int(np.round(params[2]))
    k_vote = int(np.round(params[3]))
    
    enhanced_data = nice.run(XXXX-1, label, n, x, k_ir, k_vote)
    
    if isinstance(enhanced_data, pd.DataFrame):
        enhanced_data = enhanced_data.values

    X, y = enhanced_data[:, :-1], enhanced_data[:, -1]
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3)
    clf = RandomForestClassifier(n_estimators=100)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)
    gmean = balanced_accuracy_score(y_val, y_pred)
    return -gmean  

In [ ]:
import numpy as np
from joblib import Parallel, delayed  

def BKA(objective_func, bounds, pop_size=30, max_iter=50, n_jobs=4):
    dim = len(bounds)
    lb = np.array([b[0] for b in bounds])
    ub = np.array([b[1] for b in bounds])
    
    # Initialize the population
    population = np.random.uniform(lb, ub, (pop_size, dim))
    fitness = np.array(Parallel(n_jobs=n_jobs)(delayed(objective_func)(ind) for ind in population))
    best_idx = np.argmin(fitness)
    leader = population[best_idx].copy()
    leader_fitness = fitness[best_idx]
    
    # Early stopping mechanism
    no_improve_count = 0
    prev_best = leader_fitness
    
    for t in range(max_iter):
        n = 0.05 * np.exp(-2 * (t / max_iter) ** 2)  
        p_explore = 0.9 * (1 - t / max_iter) 
        
        # Attack behavior
        new_pop = []
        for i in range(pop_size):
            r = np.random.rand()
            if r < p_explore:  # Hover exploration
                new_ind = population[i] + n * (1 + np.sin(r)) * population[i]
            else:              # Dive development
                new_ind = population[i] * (n * (2 * r - 1) + 1)
            new_pop.append(np.clip(new_ind, lb, ub))
        
        # Migration behavior
        cauchy_noise = np.random.standard_cauchy(size=(pop_size, dim))
        for i in range(pop_size):
            r_mig = np.random.rand()
            m = 2 * np.sin(r_mig + np.pi/2)  # Exploration intensity factor
            s = np.random.randint(0, pop_size)  # Randomly select comparison individuals
            
            if fitness[i] < fitness[s]:  
                new_pop[i] += cauchy_noise[i] * (leader - new_pop[i])
            else:
                new_pop[i] += cauchy_noise[i] * (leader - m * new_pop[i])
            
            new_pop[i] = np.clip(new_pop[i], lb, ub)  # Update individuals and leaders in real-time
            
            # Update individuals and leaders in real-time
            new_fit = objective_func(new_pop[i])
            if new_fit < fitness[i]:
                fitness[i] = new_fit
                population[i] = new_pop[i].copy()
                if new_fit < leader_fitness:
                    leader = new_pop[i].copy()
                    leader_fitness = new_fit
                    no_improve_count = 0
        
        # Early stopping check
        if no_improve_count >= 10 and t > 20:
            print(f"Early stopping at iteration {t}")
            break
        elif leader_fitness < prev_best:
            prev_best = leader_fitness
        else:
            no_improve_count += 1
            
    # Return the highest G-mean and the corresponding parameters
    print(f"Best G-mean: {-leader_fitness:.4f}")    
    
    return leader, leader_fitness

In [ ]:

# Initialize the BKA optimizer
best_params, best_score = BKA(
    objective_func=evaluate_params, 
    bounds=bounds,
    pop_size=30,
    max_iter=50,
    n_jobs=1
)

# Format the output results
formatted_params = [
    int(np.round(best_params[0])),  # n
    np.round(best_params[1], 2),   # x
    int(np.round(best_params[2])),  # k_ir
    int(np.round(best_params[3]))   # k_vote
]

print(f"Optimal parameters:n={formatted_params[0]}, x={formatted_params[1]:.2f}, k_ir={formatted_params[2]}, k_vote={formatted_params[3]}")
print(f"The highest G-Mean:{-best_score:.4f}")